In [1]:
!pip install pandas numpy scikit-learn xgboost

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

import joblib

# ========================
# DATA GENERATION (since no dataset provided)
# ========================

np.random.seed(42)

df = pd.DataFrame({
    "tenure_months": np.random.randint(1, 60, 500),
    "monthly_spend": np.random.uniform(20, 200, 500),
    "support_tickets": np.random.randint(0, 10, 500),
    "last_login_days": np.random.randint(0, 30, 500),
    "churn": np.random.randint(0, 2, 500)
})

print("=== Data Ingestion ===")
print(f"Loaded {df.shape[0]} records ({df.shape[1]} features)")

# ========================
# FEATURE ENGINEERING
# ========================

df["avg_monthly_spend"] = df["monthly_spend"] / df["tenure_months"]
df["engagement_score"] = df["tenure_months"] / (df["support_tickets"] + 1)

print("Engineered new features")

# ========================
# SPLIT
# ========================

X = df.drop("churn", axis=1)
y = df["churn"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# ========================
# MODELS
# ========================

models = {
    "Logistic Regression": LogisticRegression(),
    "Random Forest": RandomForestClassifier(),
    "SVM": SVC()
}

results = {}

print("\n=== Model Comparison ===")

for name, model in models.items():
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", model)
    ])

    scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring="f1")
    results[name] = scores.mean()

    print(f"{name}: F1 = {scores.mean():.3f}")

# ========================
# BEST MODEL
# ========================

best_model_name = max(results, key=results.get)
print(f"\nBest Model: {best_model_name}")

# Train best model
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", models[best_model_name])
])

pipeline.fit(X_train, y_train)

# Save model
joblib.dump(pipeline, "model.pkl")

print("\nModel saved as model.pkl")

=== Data Ingestion ===
Loaded 500 records (5 features)
Engineered new features

=== Model Comparison ===
Logistic Regression: F1 = 0.485
Random Forest: F1 = 0.472
SVM: F1 = 0.506

Best Model: SVM

Model saved as model.pkl


In [3]:
import joblib
joblib.dump(model, "churn_model.pkl")

['churn_model.pkl']